# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

**Building X (features) and y (label) strictly from ML-04's contract — nothing crosses the line.**

`y = ctr_gap_last30` is rebuilt here from `last30` fields, same as ML-04/ML-07. `X` is built
*only* from ML-04's Features bucket: `prev30` history + static SEO/content metadata. Nothing
from the Label bucket (`*_last30`) or Excluded bucket (`*_90d`, product/provider flags) goes
into `X` — enforced by an explicit assertion in Section 3, not just a promise.


In [1]:
import pandas as pd
import numpy as np

DATA_DIR = "../data"

fact = pd.read_parquet(f"{DATA_DIR}/fact_content_query_90d.parquet")
dc = pd.read_parquet(f"{DATA_DIR}/dim_content.parquet")


# ============================================================
# 1. LABEL: ctr_gap_last30
#    Built ONLY from last30 fields, matching ML-04
# ============================================================

# Actual CTR over the last 30 days
fact["ctr_last30"] = (
    fact["clicks_last30"] /
    fact["impressions_last30"].replace(0, np.nan)
)

# Position buckets
bins = [0, 3, 5, 10, 20, 50, 1000]
pos_labels = ["1-3", "3-5", "5-10", "10-20", "20-50", "50+"]

fact["pos_bucket_last30"] = pd.cut(
    fact["avg_position_last30"],
    bins=bins,
    labels=pos_labels
).astype(str)

# Expected CTR for each position bucket
expected_ctr = fact.groupby(
    "pos_bucket_last30",
    observed=True
).apply(
    lambda g: (
        g["clicks_last30"].sum() /
        g["impressions_last30"].sum()
    )
)

fact["expected_ctr_last30"] = (
    fact["pos_bucket_last30"]
    .map(expected_ctr)
    .astype(float)
)

# Target
fact["ctr_gap_last30"] = (
    fact["expected_ctr_last30"] -
    fact["ctr_last30"]
)


# ============================================================
# 2. FEATURES
#    ONLY ML-04 Features bucket
# ============================================================

FACT_FEATURES = [
    "query_char_count",
    "query_token_count",
    "impressions_prev30",
    "clicks_prev30",
    "avg_position_prev30",
    "content_visible_query_count",
    "rare_query_count",
    "rare_impressions_share"
]

DC_FEATURES = [
    "search_volume",
    "competition",
    "cpc",
    "backlinks",
    "char_count",
    "word_count"
]


# These are kept for the leakage test in Section 3.
# They are NOT included in X.
LABEL_SUSPECTS = [
    "impressions_last30",
    "clicks_last30",
    "avg_position_last30"
]


# ============================================================
# 3. FILTER LIVE CONTENT
# ============================================================

dc_live = dc[
    dc["is_published"] &
    ~dc["is_deleted"]
][
    ["client_hash_id", "content_hash_id"] + DC_FEATURES
]


# ============================================================
# 4. MERGE FACT + CONTENT DATA
# ============================================================

df = fact[
    [
        "client_hash_id",
        "content_hash_id",
        "query_hash_id"
    ]
    + FACT_FEATURES
    + LABEL_SUSPECTS
    + ["ctr_gap_last30"]
].merge(
    dc_live,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

# Remove rows where target is unavailable
df = df.dropna(subset=["ctr_gap_last30"])


# ============================================================
# 5. BUILD X AND y
# ============================================================

X = df[FACT_FEATURES + DC_FEATURES].copy()
y = df["ctr_gap_last30"].copy()


# ============================================================
# 6. MISSING-VALUE HANDLING
# ============================================================

for col in X.columns:
    if X[col].isnull().any():

        # Keep information about whether the original value
        # was missing
        X[col + "_was_missing"] = (
            X[col].isnull().astype(int)
        )

        # Fill missing values with the median
        X[col] = X[col].fillna(X[col].median())


# ============================================================
# 7. BASIC CHECKS
# ============================================================

print("Rows:", len(df))
print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nFeature columns:")
print(X.columns.tolist())

print("\nFeature dtypes:")
print(X.dtypes)

print("\nTarget summary:")
print(y.describe())

Rows: 1814458
X shape: (1814458, 21)
y shape: (1814458,)

Feature columns:
['query_char_count', 'query_token_count', 'impressions_prev30', 'clicks_prev30', 'avg_position_prev30', 'content_visible_query_count', 'rare_query_count', 'rare_impressions_share', 'search_volume', 'competition', 'cpc', 'backlinks', 'char_count', 'word_count', 'avg_position_prev30_was_missing', 'search_volume_was_missing', 'competition_was_missing', 'cpc_was_missing', 'backlinks_was_missing', 'char_count_was_missing', 'word_count_was_missing']

Feature dtypes:
query_char_count                     int64
query_token_count                    int64
impressions_prev30                   int64
clicks_prev30                        int64
avg_position_prev30                float64
content_visible_query_count          int64
rare_query_count                     int64
rare_impressions_share             float64
search_volume                      float64
competition                        float64
cpc                           

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

**What each feature means, and whether it's known before the `last30` decision point:**

| Feature | Means | Available before `last30`? |
|---|---|---|
| `query_char_count` / `query_token_count` | Length of the query text | Yes — static text property |
| `impressions_prev30` / `clicks_prev30` / `avg_position_prev30` | Performance in the 30 days *before* the label window | Yes — by construction, this is the prior period |
| `content_visible_query_count` | How many distinct queries surface this content | Yes — describes the page's reach, not `last30` performance |
| `rare_query_count` / `rare_impressions_share` | How much of the page's traffic is long-tail/rare queries | Yes — same basis as above |
| `search_volume` / `competition` / `cpc` | External keyword-market metadata (not GSC-derived) | Yes — independent of this window entirely |
| `backlinks` | Inbound link count, from `dim_content` | Yes — content-level snapshot attribute |
| `char_count` / `word_count` | Content length | Yes — static content attribute |

Missing-value rate and dtype per feature — **run this and use the real numbers to double-check
the table above, don't just trust the claim:**


In [2]:
notes = pd.DataFrame({
    "feature": FACT_FEATURES + DC_FEATURES,
    "dtype": [df[c].dtype for c in FACT_FEATURES + DC_FEATURES],
    "pct_missing_before_fill": [df[c].isnull().mean() for c in FACT_FEATURES + DC_FEATURES],
})
print(notes.to_string(index=False))

# If any pct_missing is high (say >20%), that's worth a sentence in the table above —
# a median fill on a mostly-missing column is doing a lot of guessing.


                    feature   dtype  pct_missing_before_fill
           query_char_count   int64                 0.000000
          query_token_count   int64                 0.000000
         impressions_prev30   int64                 0.000000
              clicks_prev30   int64                 0.000000
        avg_position_prev30 float64                 0.092007
content_visible_query_count   int64                 0.000000
           rare_query_count   int64                 0.000000
     rare_impressions_share float64                 0.000000
              search_volume float64                 0.007421
                competition float64                 0.007421
                        cpc float64                 0.007421
                  backlinks float64                 0.306228
                 char_count float64                 0.140001
                 word_count float64                 0.140001


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

**Attacking `X` on all three leakage types from the leakage-hunting skill, not just asserting it's clean.**

1. **Timeline / overlapping windows** — asserted in code: no column in `X` starts with a
   `last30`-window field name, and none of the `_90d` aggregates (which contain `last30`) made
   it in either.
2. **Label-derived features** — deliberately added `impressions_last30`, `clicks_last30`, and
   `avg_position_last30` into a *copy* of `X` and compared correlation with `y` before/after.
   Expect this to jump toward ~1.0, because `y` is literally computed from those three columns —
   if it doesn't, the test itself is broken, not the leakage claim.
3. **Product/decision flags** — none of `provider_used`, `model_used`, `content_updated_date`,
   or `last_optimized_date` are in `X`. These describe actions already taken by FlyRank's own
   system, not the world — using them would mean learning FlyRank's old decisions, not new
   signal.

**Run the cell below and read the real correlation numbers before writing a verdict** — if the
top of the "with leaky features" list isn't near 1.0, don't move on until you understand why;
that would mean the test harness is broken, not that the label is secretly clean.


In [3]:
# 1. Timeline check — hard assertion, not a promise
EXCLUDED_FIELDS = {"impressions_90d", "clicks_90d", "avg_position_90d", "content_total_impressions_90d",
                    "provider_used", "model_used", "anonymized_impressions_share",
                    "content_updated_date", "last_optimized_date"}

leaked = (set(LABEL_SUSPECTS) | EXCLUDED_FIELDS) & set(X.columns)
assert not leaked, f"LEAKAGE: these columns are in X and shouldn't be: {leaked}"
print("Timeline check passed — no label or excluded-window columns in X.")
print()

# 2. Label-derived-feature test: correlation WITHOUT the suspects vs WITH them
corr_clean = X.corrwith(y).sort_values(key=abs, ascending=False)
print("Correlation with y, current (clean) X:")
print(corr_clean.head(10))
print()

X_leaky = X.copy()
for col in LABEL_SUSPECTS:
    X_leaky[col] = df[col].values

corr_leaky = X_leaky.corrwith(y).sort_values(key=abs, ascending=False)
print("Correlation with y, AFTER deliberately adding the last30 fields back in:")
print(corr_leaky.head(10))
print()
print("If the leaky version's top correlation isn't dramatically higher than the clean")
print("version's, the harness is broken — fix that before trusting anything else here.")


Timeline check passed — no label or excluded-window columns in X.

Correlation with y, current (clean) X:
clicks_prev30                     -0.036494
char_count_was_missing             0.017283
word_count_was_missing             0.017283
rare_impressions_share             0.015946
avg_position_prev30                0.014419
query_char_count                   0.012349
backlinks_was_missing              0.010818
query_token_count                  0.010394
avg_position_prev30_was_missing    0.008391
competition                       -0.003497
dtype: float64

Correlation with y, AFTER deliberately adding the last30 fields back in:
clicks_last30            -0.167932
clicks_prev30            -0.036494
char_count_was_missing    0.017283
word_count_was_missing    0.017283
avg_position_last30       0.016506
rare_impressions_share    0.015946
avg_position_prev30       0.014419
query_char_count          0.012349
backlinks_was_missing     0.010818
query_token_count         0.010394
dtype: float64


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

**Excluded fields, carried over from ML-04's contract — same reasons, restated here so this
notebook stands on its own:**

| Field | Why excluded |
|---|---|
| `impressions_90d`, `clicks_90d`, `avg_position_90d`, `content_total_impressions_90d` | Span the full 90 days, which numerically contains the `last30` label window — using them would leak the label into the input |
| `provider_used`, `model_used` | Backend/operational detail, not a signal about ranking quality |
| `anonymized_impressions_share` | Could skew CTR math from unresolvable/anonymized searches |
| `content_updated_date`, `last_optimized_date` | Decision-derived (reflect an action FlyRank's own system already took) — a candidate baseline to beat, never an input, per the leakage skill's "product flags" rule |
| `client_hash_id`, `content_hash_id`, `query_hash_id`, `keyword_hash_id`, `url_hash_id` | Identifiers, not signal — kept as join keys / context, never fed to a model |


In [4]:
excluded = pd.DataFrame([
    ("impressions_90d / clicks_90d / avg_position_90d / content_total_impressions_90d",
     "Contains the last30 label window — leakage"),
    ("provider_used / model_used", "Backend/operational, not a ranking signal"),
    ("anonymized_impressions_share", "Could skew CTR from unresolvable searches"),
    ("content_updated_date / last_optimized_date", "Decision-derived — reflects FlyRank's own past action"),
    ("*_hash_id columns", "Identifiers/join keys, not model input"),
], columns=["field", "reason_excluded"])
excluded

,field,reason_excluded
0,impressions_90d / clicks_90d / avg_position_90...,Contains the last30 label window — leakage
1,provider_used / model_used,"Backend/operational, not a ranking signal"
2,anonymized_impressions_share,Could skew CTR from unresolvable searches
3,content_updated_date / last_optimized_date,Decision-derived — reflects FlyRank's own past...
4,*_hash_id columns,"Identifiers/join keys, not model input"


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Timeline check (Section 3, cell 1) actually passes — not just asserted, run and confirmed
- [ ] Ran the with/without leaky-feature correlation test and the jump matches expectations
- [ ] No product/decision flags (`content_updated_date`, `provider_used`, etc.) anywhere in `X`
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
